In [9]:
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http import models

# 1. Conexión a Qdrant con tu API Key
client = QdrantClient(
    url="http://169.58.96.176:6333",
    api_key="d2e5c0e46beff3fdeaaed224ea03696636a079f25a82ca2e70f6fb7f9fddd325",
    prefer_grpc=False,
)

collection_name = "izipay_transacciones_fraude"

# 2. Recrear la colección limpia
if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=3, distance=models.Distance.COSINE),
)

# 3. Generar puntos: 2000 transacciones (1800 legítimas y 200 fraudes)
np.random.seed(42)
n_samples = 2000
n_frauds = 200
points = []

for i in range(n_samples):
    # Los primeros 200 puntos serán fraudes (1) y los otros 1800 legítimas (0)
    is_fraud = 1 if i < n_frauds else 0

    if is_fraud:
        # Clúster de fraudes agrupado en una zona específica del espacio 3D
        vector = [
            float(np.random.uniform(0.5, 1.0)),
            float(np.random.uniform(0.5, 1.0)),
            float(np.random.uniform(0.5, 1.0)),
        ]
    else:
        # Clúster de transacciones legítimas distribuido en otra zona
        vector = [
            float(np.random.uniform(-1.0, 0.2)),
            float(np.random.uniform(-1.0, 0.2)),
            float(np.random.uniform(-1.0, 0.2)),
        ]

    points.append(
        models.PointStruct(
            id=i + 1,
            vector=vector,
            payload={
                "es_fraude": is_fraud,
                "id_transaccion": 500000 + i,
                "tipo": (
                    "Fraude Detectado" if is_fraud else "Transacción Legítima"
                ),
                "_dataset": "Izipay",
                "region": "Lima",
                "provincia": "Lima",
                "anio": "2026",
                "total_atenciones": 1,
                "resumen": (
                    "Alerta de patrón anómalo"
                    if is_fraud
                    else "Operación en rango normal"
                ),
            },
        )
    )

# 4. Subir los 2000 vectores a Qdrant en lotes de 1000
batch_size = 1000
for i in range(0, len(points), batch_size):
    client.upsert(
        collection_name=collection_name,
        points=points[i : i + batch_size]
    )

print(f"¡{n_samples} vectores cargados con éxito en '{collection_name}'!")

/tmp/ipykernel_1926995/2583942823.py:6: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


¡2000 vectores cargados con éxito en 'izipay_transacciones_fraude'!
